<a href="https://colab.research.google.com/github/timroman234/boston-voice-clone-ml/blob/main/Machin_Minimal_Viable_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1: Environment Setup & Library Installation**

In [ ]:
# ==============================================================================
# STAGE 0: ENVIRONMENT PREPARATION (F5-TTS & DEMUCS)
# ==============================================================================
print("[1/4] Installing production audio ML libraries...")

# Install F5-TTS (Zero-shot voice cloning) and Demucs (Stem separation)
!pip install -q f5-tts demucs librosa soundfile torchaudio yt-dlp

print("✅ Setup complete! Environment dependencies ready.")

**Cell 2: Audio Ingestion & Stem Isolation (Demucs)**

In [ ]:
# ==============================================================================
# STAGE 1: DIRECT AUDIO INGESTION & DEMIXING
# ==============================================================================
import os
from google.colab import files

print("--- STAGE 1: AUDIO INGESTION ---")
print("Upload any MP3 or WAV audio clip (10 to 30 seconds):")

# Interactive File Upload Widget
uploaded = files.upload()

if uploaded:
    input_filename = list(uploaded.keys())[0]
    print(f"\n[✓] Successfully uploaded: {input_filename}")

    # Run HTDemucs to isolate clean vocals from instruments/noise
    print(f"\n--- RUNNING DEMUCS SOURCE SEPARATION ON: {input_filename} ---")
    !demucs --two-stems=vocals "{input_filename}" -o ./stems

    # Get clean stem folder name
    stem_folder = os.path.splitext(input_filename)[0]
    vocal_stem_path = f"./stems/htdemucs/{stem_folder}/vocals.wav"

    print("\n" + "="*60)
    print(f"✅ STAGE 1 COMPLETE: Isolated vocal stem created!")
    print(f"📍 Vocal Stem Location: {vocal_stem_path}")
    print("="*60)
else:
    print("❌ No file uploaded. Please run the cell again and select an audio file.")

**Cell 3: Audio Preprocessing & Zero-Shot Voice Conditioning**

In [ ]:
# ==============================================================================
# STAGE 2 & 3: ZERO-SHOT VOICE SYNTHESIS (F5-TTS)
# ==============================================================================
import torch
from f5_tts.api import F5TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Running inference on: {device.upper()}")

print("\n[Loading F5-TTS zero-shot model into VRAM...]")
f5tts = F5TTS()

# 1. MUST match the words spoken/sung in your YouTube 20-second sample!
# Example: If your YouTube sample says "Pretty woman walking down the street", put that here:
ref_text = "I looked out this morning and the sun was gone. Turned on some music to start my day. I lost myself in a familiar song. I closed my eyes and I slipped away. Aaaaaaa"

# 2. The new prompt you want the cloned voice to say:
gen_text = "Boston is an American rock band formed in 1975 in Boston, Massachusetts, by chief songwriter and composer Tom Scholz."

output_path = "./cloned_artist_vocal.wav"

print("\n--- STAGE 3: GENERATING CONDITIONED VOICE CLONE ---")
print(f"Reference Text : '{ref_text}'")
print(f"Target Prompt  : '{gen_text}'")

wav, sr, spect = f5tts.infer(
    ref_file=vocal_stem_path,
    ref_text=ref_text,
    gen_text=gen_text,
    file_wave=output_path
)

print("\n" + "="*60)
print(f"✅ STAGE 3 COMPLETE: Synthetic audio successfully rendered from YouTube voice!")
print(f"📍 Output file: {output_path}")
print("="*60)

**Cell 4: Audio Playback & Automated Quality Assurance (QA Metrics)**

In [ ]:
# ==============================================================================
# STAGE 4: AUDIO PLAYBACK & QUALITY METRICS
# ==============================================================================
import IPython.display as ipd
import librosa
import numpy as np

print("--- STAGE 4: AUDIO PLAYBACK & QUALITY METRICS ---")

# Playback
print("\n1. Isolated Reference Vocal Stem (Demucs output):")
ipd.display(ipd.Audio(vocal_stem_path))

print("\n2. Generated AI Voice Clone (F5-TTS output):")
ipd.display(ipd.Audio(output_path))

# Compute signal metrics
y, sr = librosa.load(output_path, sr=None)
rms_energy = np.mean(librosa.feature.rms(y=y))
zcr_val = np.mean(librosa.feature.zero_crossing_rate(y=y))

print("\n" + "="*60)
print("📊 AUTOMATED AUDIO QUALITY REPORT")
print("="*60)
print(f"• Sample Rate        : {sr} Hz")
print(f"• Audio Duration     : {len(y)/sr:.2f} seconds")
print(f"• Perceived Energy   : {rms_energy:.4f}")
print(f"• Zero Crossing Rate : {zcr_val:.4f}")
print("="*60)

In [ ]:
# ==============================================================================
# VISUALIZATION: MEL-SPECTROGRAM ANALYSIS
# ==============================================================================
# ==============================================================================
# VISUALIZATION: MEL-SPECTROGRAM ANALYSIS (BULLETPROOF FIX)
# ==============================================================================
%matplotlib inline
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

print("🎨 Rendering Mel-Spectrograms...")

# Fallback path verification
ref_path = vocal_stem_path if 'vocal_stem_path' in globals() and os.path.exists(vocal_stem_path) else "./stems/htdemucs/sample_input/vocals.wav"
gen_path = output_path if 'output_path' in globals() and os.path.exists(output_path) else "./cloned_artist_vocal.wav"

if not os.path.exists(ref_path) or not os.path.exists(gen_path):
    print("❌ Audio files not found. Make sure Stage 1 and Stage 3 have finished running!")
else:
    # 1. Load audio files
    y_ref, sr_ref = librosa.load(ref_path, sr=24000)
    y_gen, sr_gen = librosa.load(gen_path, sr=24000)

    # 2. Compute Mel-spectrograms
    S_ref_db = librosa.power_to_db(librosa.feature.melspectrogram(y=y_ref, sr=sr_ref, n_mels=128), ref=np.max)
    S_gen_db = librosa.power_to_db(librosa.feature.melspectrogram(y=y_gen, sr=sr_gen, n_mels=128), ref=np.max)

    # 3. Create Figure
    plt.close('all') # Clear previous figure cache
    fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    # Top Plot: Reference Vocal
    img1 = librosa.display.specshow(S_ref_db, x_axis='time', y_axis='mel', sr=sr_ref, ax=ax[0], cmap='magma')
    ax[0].set_title('1. YouTube Reference Vocal Stem (Demucs Output)', fontsize=11, fontweight='bold')
    fig.colorbar(img1, ax=ax[0], format='%+2.0f dB')

    # Bottom Plot: Generated Voice Clone
    img2 = librosa.display.specshow(S_gen_db, x_axis='time', y_axis='mel', sr=sr_gen, ax=ax[1], cmap='magma')
    ax[1].set_title('2. Generated AI Voice Clone (F5-TTS Output)', fontsize=11, fontweight='bold')
    fig.colorbar(img2, ax=ax[1], format='%+2.0f dB')

    plt.tight_layout()
    plt.show()
    print("✅ Spectrogram comparison successfully generated!")